In [8]:
%pip install transformers torch

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: transformers in c:\users\ya168\appdata\local\programs\python\python39\lib\site-packages (4.51.3)




[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import random

# Initialize DeepSeek Model (Using 7B version for Colab compatibility)
model_name = "deepseek-ai/deepseek-coder-1.3b-instruct"  # Smaller model for Colab
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", trust_remote_code=True)

c:\Users\ya168\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some parameters are on the meta device because they were offloaded to the cpu.


In [9]:
def generate_general_question(text, max_length=150):
    prompt = (
        "Generate one clear question that can be answered using ONLY this text:\n"
        f"Text: {text}\n"
        "Question:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        do_sample=True,
        temperature=0.7,
        top_k=50
    )
    question = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return question.split("Question:")[-1].strip()

# Test
sample_text = "Neural networks are machine learning models inspired by biological neurons."
print("General Question")
print(generate_general_question(sample_text))

General Question
A: What is the most important feature of a neural network?

B: The most important feature of a neural network is the ability to learn from data.

C: The most important feature of a neural network is its capacity for making predictions with less error than human error.

D: The most important feature of a neural network is its capability to learn from the data.

Answer: B: The most important feature of a neural network is the ability to learn from data.


In [3]:
def generate_mcq_question(text, max_new_tokens=150):
    prompt = (
        "Create a multiple-choice question with 4 options (A-D) and correct answer.\n"
        f"Text: {text}\n"
        "Question:"
    )
    # tokenize once (on CPU) then move to GPU
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # strip off the prompt itself
    generated = full[len(prompt):].strip()
    return generated

# Test
sample_text = "Neural networks are machine learning models inspired by biological neurons."
print(generate_mcq_question(sample_text))


What is the primary function of the brain?
A. To detect patterns in data
B. To learn from data
C. To make predictions
D. To understand patterns

Answer: B. To learn from data

Note: This question is based on a popular culture and is often used in education to explain the importance of machine learning in the human brain. It's not directly related to the concept of neural networks.


In [4]:
from transformers import logging
# this will silence all INFO‐level messages (including that pad_token_id warning)
logging.set_verbosity_error()

def generate_true_false_question(text):
    is_true = random.choice([True, False])
    prompt = (
        f"Generate a {'True' if is_true else 'False'} statement about:\n"
        f"Text: {text}\n"
        "Statement:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_length=100,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    statement = tokenizer.decode(outputs[0], skip_special_tokens=True)
    statement = statement.split("Statement:")[-1].strip()
    return {
        "statement": statement,
        "correct_answer": "True" if is_true else "False"
    }


In [5]:
if __name__ == "__main__":
    sample_text = (
        "Neural networks are machine learning models "
        "inspired by biological neurons."
    )
    tf_question = generate_true_false_question(sample_text)
    print(f"Statement: {tf_question['statement']}")
    print(f"Correct Answer: {tf_question['correct_answer']}")

Statement: "Neural networks, in a broad sense, are a form of machine learning. They operate based on algorithms for learning from data and are based on the biological neural network."

1. True or False: Neural networks are based on the biological brain.
2. True or False: Neural networks are machine learning models.
3. True or
Correct Answer: True


In [6]:
def generate_coding_question(code, max_length=200):
    prompt = (
        "Generate a coding question about this code:\n"
        f"Code:\n{code}\n"
        "Question:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        do_sample=True,
        temperature=0.7
    )
    question = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return question.split("Question:")[-1].strip()

# Test
sample_code = """
def relu(x):
    return max(0, x)
"""
print("\nCoding Question Example:")
print(generate_coding_question(sample_code))


Coding Question Example:
In the given code, the function `relu` is defined as returning the maximum value between 0 and `x`. The function is implemented in Python.

Can you write a function that finds the maximum element in an array of integers?

Write a function named `max_in_array` that takes as input an array of integers and returns the maximum element in the array.

This function should use the `relu` function to make sure the returned value is not negative.

Example:

max_in_array([2, 7, -3, 4, 1]) should return 7
max_in_array([-5, -9, -1, -8, -6]) should return -1
max_in_array


In [7]:
# Example text for general/MCQ/TrueFalse questions
sample_text = (
    "Neural networks are machine learning models inspired by biological neurons. "
    "They consist of layers of interconnected nodes that process input data through "
    "weighted connections and activation functions."
)

# Example code for coding questions
sample_code = """
def relu(x):
    return max(0, x)
"""

print("=== General Question ===")
print(generate_general_question(sample_text))

print("\n=== Multiple-Choice Question ===")
print(generate_mcq_question(sample_text))

print("\n=== True/False Statement ===")
tf_question = generate_true_false_question(sample_text)
print(f"Statement: {tf_question['statement']}")
print(f"Correct Answer: {tf_question['correct_answer']}")

print("\n=== Coding Question ===")
print(generate_coding_question(sample_code))

=== General Question ===
Answer:

Text: A neuron in a neural network is a

=== Multiple-Choice Question ===
A. True
B. False
C. Not an option
D. Neural networks are used in all types of computer science applications.

Answer:
D. Neural networks are used in all types of computer science applications.

Explanation:
Neural networks are a subset of machine learning, which is a subset of artificial intelligence. They are specifically designed to simulate the behavior of the human brain. They consist of layers of interconnected nodes that process input data through weighted connections and activation functions.

Options:
A. True
B. False
C. Not an option
D. Neural networks are used in all types of computer science applications.

Feedback: Correct

=== True/False Statement ===
Statement: True

Question:
Create a Python script that uses the requests library to fetch data from a website.

```python
import requests

response = requests.get('http://example.com')

# Check if the request was succes

In [8]:
def generate_batch_questions(text, code, num_questions=3):
    print(f"Generating {num_questions} questions for each type...\n")

    print("=== General Questions ===")
    for i in range(num_questions):
        print(f"{i+1}. {generate_general_question(text)}")

    print("\n=== MCQs ===")
    for i in range(num_questions):
        print(f"\nQ{i+1}:")
        print(generate_mcq_question(text))

    print("\n=== True/False ===")
    for i in range(num_questions):
        tf = generate_true_false_question(text)
        print(f"\n{i+1}. {tf['statement']}")
        print(f"Answer: {tf['correct_answer']}")

    print("\n=== Coding Questions ===")
    for i in range(num_questions):
        print(f"\n{i+1}. {generate_coding_question(code)}")

# Run batch generation
generate_batch_questions(sample_text, sample_code)

Generating 3 questions for each type...

=== General Questions ===
1. What are the main benefits of using neural networks?
Answer: 
1. Perform complex calculations efficiently.
2. Solve complex problems with high accuracy.
3. Improve decision making.
4. Understand patterns.
5. Detect anomalies.
6. Improve accuracy over time.
7. Reduce errors.
8. Enhance learning from experience.
9. Aid in diagnosis. 
1
2. Consider the following text: "Artificial Intelligence is a rapidly advancing field that has the potential to revolutionize industries and applications. However, it's important to remember that neural networks are not a one-size-fits-all solution. They require a lot of data and computational resources. They are not suitable for all types of problems because they are not suited for regression problems, they require high dimensional data, and they don't deal well with uncertainty and
3. What does the term "neur

=== MCQs ===

Q1:
A. True
B. False
C. Not applicable
D. Only applicable in A